# FMODetect-v2 — Colab training notebook

PyTorch port of **FMODetect** (Rozumnyi et al., 2021) with three novelty axes:

1. **CBAM attention** at every U-Net block
2. **Joint TDF + matting** multi-task head (one network, two decoders)
3. **Uncertainty-weighted boundary loss** (Gaussian-NLL + L1 on ∇D)

**Hardware**: requires a GPU runtime. Free T4 (16 GB) is plenty; A100/L4 will be faster.

**Pipeline**: clone repo → install deps → download VOT2016 (backgrounds) + falling_objects (eval) → generate 5k synthetic FMO samples → train → eval → save checkpoint to Drive.

**Source**: <https://github.com/jai-krishna-0921/FMODetect-v2>


In [1]:
#@title FMODetect-v2 — verify GPU
# If this errors with "nvidia-smi: not found", you're on a CPU runtime.
# Switch via: Runtime ▸ Change runtime type ▸ T4 GPU (free) or L4/A100 (paid)
import shutil, torch
if shutil.which("nvidia-smi") is None:
    raise SystemExit(
        "❌ No GPU runtime detected.\n"
        "   In Colab menu: Runtime → Change runtime type → 'T4 GPU' (free tier is fine).\n"
        "   Then re-run this cell."
    )
import subprocess
print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
     "--format=csv,noheader"]).decode().strip())
print("torch:", torch.__version__, " cuda?", torch.cuda.is_available(),
      " device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
free, total = torch.cuda.mem_get_info() if torch.cuda.is_available() else (0, 0)
print(f"vram free: {free//1024//1024} / {total//1024//1024} MB")


Tesla T4, 15360 MiB, 580.82.07, 7.5
torch: 2.10.0+cu128  cuda? True  device: Tesla T4
vram free: 14807 / 14912 MB


In [2]:
#@title Mount Google Drive (for dataset + checkpoint persistence)
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/FMODetect-v2"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/datasets", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/experiments", exist_ok=True)
print("Drive root:", DRIVE_ROOT)


Mounted at /content/drive
Drive root: /content/drive/MyDrive/FMODetect-v2


In [3]:
#@title Clone the repo and `cd` into it
import os, subprocess
REPO_URL = "https://github.com/jai-krishna-0921/FMODetect-v2.git"
WORK = "/content/FMODetect-v2"
if not os.path.exists(WORK):
    subprocess.check_call(["git", "clone", REPO_URL, WORK])
else:
    subprocess.check_call(["git", "-C", WORK, "pull"])
os.chdir(WORK)
print("cwd:", os.getcwd())
print(subprocess.check_output(["git", "log", "-1", "--oneline"]).decode())


cwd: /content/FMODetect-v2
78097be Initial FMODetect-v2: PyTorch port with CBAM + multi-task + uncertainty-boundary loss



In [4]:
#@title Install Python dependencies
# Colab's torch is already CUDA-enabled; we only need the extras.
import subprocess, sys

PKGS = [
    "scikit-image>=0.25",
    "h5py>=3.12",
    "imageio[ffmpeg]>=2.36",
    "pillow>=11",
    "scipy>=1.14",
    "opencv-python-headless>=4.10",
    "mlflow>=3.5",
    "clearml>=1.16",
    "pydantic>=2.9",
    "pyyaml>=6.0",
    "typer>=0.13",
    "rich>=13.9",
    "tqdm>=4.66",
    "fastapi>=0.115",
    "uvicorn[standard]>=0.32",
    "tensorboard>=2.18",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PKGS])
print("deps installed")


deps installed


In [5]:
#@title Download VOT2016 backgrounds + falling_objects eval set
# These are ~1-2 GB total. Persisted to Drive so re-runs are instant.
import os, subprocess, json
from pathlib import Path

DATA = Path(f"{DRIVE_ROOT}/datasets")
PTAK = "https://ptak.felk.cvut.cz/personal/rozumden"
VOT  = "https://data.votchallenge.net/vot2016/main"

def curl(url, out, **_):
    out = Path(out); out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists() and out.stat().st_size > 100:
        return
    subprocess.check_call([
        "curl","-kL","-C","-","--retry","5","--retry-delay","5",
        "--retry-all-errors","--connect-timeout","30","-sS", url, "-o", str(out)
    ])

# --- VOT2016 ---
vot_root = DATA / "vot2016"
vot_root.mkdir(parents=True, exist_ok=True)
desc_p = vot_root / "description.json"
curl(f"{VOT}/description.json", desc_p)
desc = json.loads(desc_p.read_text())
seqs = desc["sequences"]
print(f"VOT2016: {len(seqs)} sequences")
for i, s in enumerate(seqs):
    name = s["name"]
    seq_dir = vot_root / name
    seq_dir.mkdir(exist_ok=True)
    # Frames
    if not (seq_dir / ".done").exists():
        zf = seq_dir / "color.zip"
        curl(f"{VOT}/{s['channels']['color']['url']}", zf)
        subprocess.check_call(["unzip","-q","-o", str(zf), "-d", str(seq_dir)])
        zf.unlink()
        (seq_dir / ".done").write_text("")
    # Annotations
    if not (seq_dir / ".ann_done").exists():
        zf = seq_dir / "annotations.zip"
        curl(f"{VOT}/{s['annotations']['url']}", zf)
        subprocess.check_call(["unzip","-q","-o", str(zf), "-d", str(seq_dir)])
        zf.unlink()
        (seq_dir / ".ann_done").write_text("")
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(seqs)} done")
print(f"VOT2016 OK. frames: {sum(1 for _ in vot_root.rglob('*.jpg'))}, ann: {len(list(vot_root.glob('*/.ann_done')))}")


VOT2016: 60 sequences
  10/60 done
  20/60 done
  30/60 done
  40/60 done
  50/60 done
  60/60 done
VOT2016 OK. frames: 21455, ann: 60


In [6]:
#@title Kick off eval downloads (falling + TbD-3D + TbD) IN BACKGROUND
# Runs all three downloads in parallel via subprocess.Popen so they don't
# block the synth gen + training cells. TbD is 25 GB; will likely still be
# running when training starts — that's fine, eval cell waits for it.
import subprocess, time
from pathlib import Path

LOG_DIR = DATA / "_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_TBD = True   # 25 GB; user requested it

def launch_zip_dl(name: str, url: str, out_root: Path):
    out_root = Path(out_root); out_root.mkdir(parents=True, exist_ok=True)
    done = out_root / ".done"
    if done.exists():
        print(f"[{name}] already done"); return None
    zf = out_root / f"{name}.zip"
    log = open(LOG_DIR / f"{name}.log", "w")
    cmd = (
        f"curl -kL -C - --retry 5 --retry-delay 5 --retry-all-errors "
        f"--connect-timeout 30 -sS {url!r} -o {str(zf)!r} && "
        f"unzip -q -o {str(zf)!r} -d {str(out_root)!r} && "
        f"rm {str(zf)!r} && touch {str(done)!r} && echo OK"
    )
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT)
    print(f"[{name}] launched PID={p.pid}, log → {LOG_DIR/f'{name}.log'}")
    return p

procs = {}
procs["falling"]  = launch_zip_dl("falling", f"{PTAK}/falling_imgs_gt.zip", DATA/"eval/falling")
procs["TbD-3D"]   = launch_zip_dl("TbD-3D",  f"{PTAK}/TbD-3D-imgs.zip",    DATA/"eval/TbD-3D")
if DOWNLOAD_TBD:
    procs["TbD"]  = launch_zip_dl("TbD",     f"{PTAK}/TbD_imgs_upd.zip",   DATA/"eval/TbD")

# Snapshot of progress after 5s
time.sleep(5)
print("\n--- 5s check ---")
for name in procs:
    p = procs[name]
    if p is None: continue
    print(f"[{name}] pid {p.pid} still running: {p.poll() is None}")
    # File size grown?
    zf = DATA / f"eval/{name}/{name}.zip"
    if zf.exists():
        mb = zf.stat().st_size // 1024 // 1024
        print(f"           current size: {mb} MB")
print("\nBackground downloads launched. They will continue while you run the next cells.")
print("Monitor any time with: !tail -f /content/drive/MyDrive/FMODetect-v2/datasets/_logs/TbD.log")


[falling] launched PID=14757, log → /content/drive/MyDrive/FMODetect-v2/datasets/_logs/falling.log
[TbD-3D] launched PID=14760, log → /content/drive/MyDrive/FMODetect-v2/datasets/_logs/TbD-3D.log
[TbD] launched PID=14763, log → /content/drive/MyDrive/FMODetect-v2/datasets/_logs/TbD.log

--- 5s check ---
[falling] pid 14757 still running: True
           current size: 58 MB
[TbD-3D] pid 14760 still running: True
           current size: 60 MB
[TbD] pid 14763 still running: True
           current size: 55 MB

Background downloads launched. They will continue while you run the next cells.
Monitor any time with: !tail -f /content/drive/MyDrive/FMODetect-v2/datasets/_logs/TbD.log


In [7]:
#@title Verify dataset annotations are present and parseable
import subprocess
out = subprocess.check_output([
    "python", "scripts/verify_annotations.py",
    "--datasets-root", str(DATA),
    "--out", str(DATA / "_annotation_report.json"),
], text=True, cwd=WORK, env={**os.environ, "PYTHONPATH": WORK})
print(out)



Annotation report written to /content/drive/MyDrive/FMODetect-v2/datasets/_annotation_report.json

  [vot2016   ] 60 sequences, 60/60 annotated  OK
  [falling   ] 0 sequences, 0 frames, 0 with GT
  [TbD-3D    ] 0 sequences, 0 frames, 0 with GT
  [TbD       ] 0 sequences, 0 frames, 0 with GT




In [17]:
#@title Generate the synthetic VOT-FMO dataset (5,000 samples) → local /content (inline; live progress)
# Inline call so tqdm renders live in this cell. Writes to local /content (fast disk),
# then mirrors final H5 to Drive for persistence across runtime resets.
import sys
sys.path.insert(0, WORK)
import importlib
import random, time
from pathlib import Path
import h5py, numpy as np
from tqdm.notebook import tqdm

# Force a re-import of the modules in case anything was patched
for m in list(sys.modules):
    if m.startswith("src.fmodetect"):
        del sys.modules[m]

from src.fmodetect.data.synthesize import (
    SynthConfig, load_background_paths, load_pattern_paths,
    read_image, read_rgba, synthesize_sample,
)
from src.fmodetect.data.patterns import generate_pattern_bank

N_SAMPLES = 5000
SHAPE = (256, 512)
LOCAL_H5 = Path("/content/synth/vot_fmo.h5")
DRIVE_H5 = DATA / "synth" / "vot_fmo.h5"
SYNTH_H5 = LOCAL_H5

LOCAL_H5.parent.mkdir(parents=True, exist_ok=True)
DRIVE_H5.parent.mkdir(parents=True, exist_ok=True)

# Resume-from-Drive shortcut
if DRIVE_H5.exists() and not LOCAL_H5.exists():
    print(f"copying existing Drive H5 → local ({DRIVE_H5.stat().st_size//1024//1024} MB)...")
    import shutil; shutil.copy(DRIVE_H5, LOCAL_H5)

if LOCAL_H5.exists():
    print(f"already exists: {LOCAL_H5}  ({LOCAL_H5.stat().st_size//1024//1024} MB)")
else:
    # Pattern bank (auto-generate if empty)
    pat_dir = DATA / "patterns"
    pat_paths = load_pattern_paths(pat_dir)
    if not pat_paths:
        print(f"generating 100 procedural patterns at {pat_dir}...")
        generate_pattern_bank(pat_dir, 100, seed=42)
        pat_paths = load_pattern_paths(pat_dir)
    bg_paths = load_background_paths(DATA / "vot2016")
    print(f"backgrounds: {len(bg_paths)}  patterns: {len(pat_paths)}")

    rng = random.Random(42)
    cfg = SynthConfig(out_shape=SHAPE)

    rejected = 0; written = 0
    t0 = time.time()
    with h5py.File(LOCAL_H5, "w") as f:
        f.attrs["n_samples_target"] = N_SAMPLES
        f.attrs["out_shape"] = list(SHAPE)
        f.attrs["seed"] = 42
        bar = tqdm(total=N_SAMPLES, desc="synth")
        while written < N_SAMPLES:
            bg = read_image(bg_paths[rng.randrange(len(bg_paths))])
            pat = read_rgba(pat_paths[rng.randrange(len(pat_paths))])
            sample = synthesize_sample(bg, pat, cfg, rng)
            if sample is None:
                rejected += 1; continue
            g = f.create_group(f"sample_{written:08d}")
            for k, v in sample.items():
                kw = {}
                if hasattr(v, "shape") and v.shape:
                    kw["compression"] = "lzf"
                g.create_dataset(k, data=v, **kw)
            written += 1
            bar.update(1)
            if written % 500 == 0:
                bar.set_postfix(rejected=rejected, mb=f"{LOCAL_H5.stat().st_size//1024//1024}")
        f.attrs["n_samples"] = written
        bar.close()
    print(f"\n[build_dataset] wrote {written} samples to {LOCAL_H5} "
          f"({rejected} rejected, {time.time()-t0:.1f}s)")

# Mirror to Drive
import shutil, time
print(f"\ncopying {LOCAL_H5}  →  {DRIVE_H5} ...")
t0 = time.time()
shutil.copy(LOCAL_H5, DRIVE_H5)
print(f"copy took {time.time()-t0:.1f}s; final size {DRIVE_H5.stat().st_size//1024//1024} MB")
print(f"H5 ready (using local): {SYNTH_H5}")


backgrounds: 21455  patterns: 100


synth:   0%|          | 0/5000 [00:00<?, ?it/s]


[build_dataset] wrote 5000 samples to /content/synth/vot_fmo.h5 (75 rejected, 4145.0s)

copying /content/synth/vot_fmo.h5  →  /content/drive/MyDrive/FMODetect-v2/datasets/synth/vot_fmo.h5 ...
copy took 194.4s; final size 16560 MB
H5 ready (using local): /content/synth/vot_fmo.h5


In [10]:
#@title Write the Colab training config (overrides default for 16 GB GPU + Drive paths)
import yaml

cfg = {
    "seed": 42,
    "device": "cuda",
    "data": {
        "h5_path": str(SYNTH_H5),
        "val_fraction": 0.04,
        "num_workers": 2,
        "pin_memory": True,
    },
    "model": {
        "in_channels": 6,
        "base_channels": [16, 64, 128, 256, 256],
        "use_cbam": True,
        "predict_matting": True,
        "predict_uncertainty": True,
    },
    "loss": {"matting_weight": 0.5, "boundary_weight": 0.1},
    "train": {
        "epochs": 40,                # capped for 5h20m Colab session; early-stop will kick in earlier
        "batch_size": 16,            # T4 has 16 GB
        "grad_accum_steps": 1,
        "lr": 2.0e-5,
        "weight_decay": 1.0e-5,
        "amp": False,                # fp16 NaNs from no-norm CBAM blocks; fp32 fits fine on T4
        "channels_last": True,
        "grad_clip_norm": 1.0,
        "log_every_n_steps": 20,
        "val_every_n_epochs": 1,
        "save_every_n_epochs": 5,
        "early_stop_patience": 12,
    },
    "logging": {
        "mlflow_uri": f"file:{DRIVE_ROOT}/experiments/mlruns",
        "mlflow_experiment": "fmodetect-v2-colab",
        "use_clearml": False,
        "clearml_project": "fmodetect-v2",
        "tensorboard_dir": f"{DRIVE_ROOT}/experiments/tb",
    },
    "checkpoints": {
        "dir": f"{DRIVE_ROOT}/experiments/checkpoints",
        "keep_best_n": 3,
    },
}
cfg_path = Path(WORK) / "configs" / "colab.yaml"
cfg_path.write_text(yaml.safe_dump(cfg))
print(f"wrote {cfg_path}")
print(yaml.safe_dump(cfg))


wrote /content/FMODetect-v2/configs/colab.yaml
checkpoints:
  dir: /content/drive/MyDrive/FMODetect-v2/experiments/checkpoints
  keep_best_n: 3
data:
  h5_path: /content/synth/vot_fmo.h5
  num_workers: 2
  pin_memory: true
  val_fraction: 0.04
device: cuda
logging:
  clearml_project: fmodetect-v2
  mlflow_experiment: fmodetect-v2-colab
  mlflow_uri: file:/content/drive/MyDrive/FMODetect-v2/experiments/mlruns
  tensorboard_dir: /content/drive/MyDrive/FMODetect-v2/experiments/tb
  use_clearml: false
loss:
  boundary_weight: 0.1
  matting_weight: 0.5
model:
  base_channels:
  - 16
  - 64
  - 128
  - 256
  - 256
  in_channels: 6
  predict_matting: true
  predict_uncertainty: true
  use_cbam: true
seed: 42
train:
  amp: false
  batch_size: 16
  channels_last: true
  early_stop_patience: 12
  epochs: 40
  grad_accum_steps: 1
  grad_clip_norm: 1.0
  log_every_n_steps: 20
  lr: 2.0e-05
  save_every_n_epochs: 5
  val_every_n_epochs: 1
  weight_decay: 1.0e-05



In [ ]:
#@title Train! Inline call (no subprocess) so tqdm + prints stream live.
import sys, importlib
sys.path.insert(0, WORK)
# Force re-import of fmodetect modules to pick up any updates
for m in list(sys.modules):
    if m.startswith("src.fmodetect"):
        del sys.modules[m]

from src.fmodetect.training.loop import train
from src.fmodetect.utils.config import load_config

cfg = load_config(f"{WORK}/configs/colab.yaml")
print("Loaded config. Training start ...")
train(cfg)


Loaded config. Training start ...
[train] device=cuda
[train] train batches=300, val batches=13
[train] model params: 6.03M


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/22 18:52:05 INFO mlflow.tracking.fluent: Experiment with name 'fmodetect-v2-colab' does not exist. Creating a new experiment.
epoch 0: 100%|██████████| 300/300 [07:19<00:00,  1.47s/it, total=-0.2909]


[train] epoch 0: train_total=0.4110 val_total=-0.6886


epoch 1: 100%|██████████| 300/300 [07:17<00:00,  1.46s/it, total=-1.3056]


[train] epoch 1: train_total=-1.1488 val_total=-1.2736


...

epoch 27: 100%|██████████| 300/300 [07:18<00:00,  1.46s/it, total=-3.5233]


[train] epoch 27: train_total=-3.5682 val_total=-3.7483


epoch 28: 100%|██████████| 300/300 [07:16<00:00,  1.46s/it, total=-3.2099]


[train] epoch 28: train_total=-3.5904 val_total=-3.7559


epoch 29:  96%|█████████▋| 289/300 [07:01<00:15,  1.45s/it, total=-3.7247]

In [16]:
#@title Wipe stale H5 files (run before re-running cell 8)
import os
for p in ["/content/synth/vot_fmo.h5",
          f"{DRIVE_ROOT}/datasets/synth/vot_fmo.h5"]:
    if os.path.exists(p):
        sz = os.path.getsize(p) // 1024 // 1024
        os.remove(p)
        print(f"deleted {p} ({sz} MB)")
    else:
        print(f"not present: {p}")


deleted /content/synth/vot_fmo.h5 (627 MB)
deleted /content/drive/MyDrive/FMODetect-v2/datasets/synth/vot_fmo.h5 (627 MB)


In [ ]:
#@title Eval — wait for any pending downloads, then run on every available dataset
import subprocess, glob, os, time
from pathlib import Path

# Wait for the background eval downloads to finish (poll Drive .done flags)
EVAL_SETS = [
    ("falling",  DATA / "eval/falling"),
    ("TbD-3D",   DATA / "eval/TbD-3D"),
    ("TbD",      DATA / "eval/TbD"),
]
deadline = time.time() + 60 * 30  # wait up to 30 min
ready = []
print("Waiting for eval dataset downloads to finish (max 30 min)...")
while time.time() < deadline:
    ready = [(n, p) for n, p in EVAL_SETS if (p / ".done").exists()]
    pending = [n for n, p in EVAL_SETS if not (p / ".done").exists()]
    if not pending:
        break
    print(f"  ready: {[n for n,_ in ready]}  pending: {pending}  (waiting...)")
    time.sleep(30)
print(f"\nUsable eval sets: {[n for n,_ in ready]}")

# Verify annotations across whatever's ready
subprocess.run(["python", "scripts/verify_annotations.py",
                "--datasets-root", str(DATA),
                "--out", str(DATA / "_annotation_report.json")],
               cwd=WORK, env={**os.environ, "PYTHONPATH": WORK}, check=False)

# Pick latest checkpoint
ckpt = sorted(glob.glob(f"{DRIVE_ROOT}/experiments/checkpoints/run_*/best.pt"))[-1]
print("\nUsing checkpoint:", ckpt)

# Run eval on each ready set
for name, p in ready:
    out_json = f"{DRIVE_ROOT}/experiments/eval_{name}.json"
    print(f"\n=== eval on {name} ===")
    subprocess.run([
        "python", "scripts/eval.py",
        "--ckpt", ckpt,
        "--dataset", str(p),
        "--out", out_json,
    ], cwd=WORK, env={**os.environ, "PYTHONPATH": WORK}, check=False)
    print(f"wrote {out_json}")


In [ ]:
#@title Inference smoke test on the original repo's example image
# Confirms the trained model + post-hoc trajectory extraction produce a plausible overlay.
import subprocess, glob, os
from IPython.display import Image as IPyImage, display

ckpt = sorted(glob.glob(f"{DRIVE_ROOT}/experiments/checkpoints/run_*/best.pt"))[-1]
# Pull the original example into Colab from the upstream repo
subprocess.run(["curl","-sL","-o","/tmp/ex1_im.png",
                "https://raw.githubusercontent.com/rozumden/FMODetect/master/example/ex1_im.png"], check=True)
subprocess.run(["curl","-sL","-o","/tmp/ex1_bgr.png",
                "https://raw.githubusercontent.com/rozumden/FMODetect/master/example/ex1_bgr.png"], check=True)

out_dir = f"{DRIVE_ROOT}/experiments/infer_smoke"
os.makedirs(out_dir, exist_ok=True)
subprocess.run([
    "python", "scripts/infer.py",
    "--ckpt", ckpt,
    "--image", "/tmp/ex1_im.png",
    "--bgr",   "/tmp/ex1_bgr.png",
    "--out", f"{out_dir}/ex1.png",
], cwd=WORK, env={**os.environ, "PYTHONPATH": WORK}, check=True)
display(IPyImage(f"{out_dir}/ex1.png"))


In [ ]:
#@title (Optional) Mid-training visualization — peek at current best checkpoint
# Run this any time while training is in flight to see the current best checkpoint's
# predicted TDF on one synthetic sample. Useful as a "is the model actually learning?" sanity check.
import glob, random, numpy as np, torch
from pathlib import Path
import matplotlib.pyplot as plt

ckpts = sorted(glob.glob(f"{DRIVE_ROOT}/experiments/checkpoints/run_*/best.pt"))
if not ckpts:
    print("No checkpoint yet — training hasn't saved one. Wait a few epochs.")
else:
    from src.fmodetect.inference.runner import load_model, infer_pair
    from src.fmodetect.data.dataset import SynthFMODataset

    ds = SynthFMODataset(str(SYNTH_H5))
    print(f"checkpoint: {ckpts[-1]}")
    print(f"dataset:    {SYNTH_H5}   ({len(ds)} samples)")

    model = load_model(ckpts[-1], device="cuda")
    idx = random.randrange(len(ds))
    s = ds[idx]
    x = s["x"].numpy()   # (6, H, W) — already normalized
    # Recover unnormalized image for display by reading raw h5 entry
    import h5py
    with h5py.File(str(SYNTH_H5), "r") as f:
        g = f[f"sample_{idx:08d}"]
        img = g["image"][()]; bg = g["bgr"][()]; gt_tdf = g["tdf"][()]; gt_hm = g["hm"][()]
    res = infer_pair(model, img, bg, device="cuda")

    fig, ax = plt.subplots(2, 3, figsize=(14, 6))
    ax[0,0].imshow(img); ax[0,0].set_title("input image")
    ax[0,1].imshow(gt_tdf, cmap="inferno"); ax[0,1].set_title("GT TDF")
    ax[0,2].imshow(gt_hm,  cmap="gray");    ax[0,2].set_title("GT matting H*M")
    ax[1,0].imshow(bg);  ax[1,0].set_title("background")
    ax[1,1].imshow(res.tdf, cmap="inferno", vmin=0, vmax=1); ax[1,1].set_title(f"pred TDF (range [{res.tdf.min():.2f}, {res.tdf.max():.2f}])")
    ax[1,2].imshow(res.hm,  cmap="gray");   ax[1,2].set_title("pred H*M")
    for a in ax.flat: a.axis("off")
    fig.suptitle(f"sample {idx}   |   {len(res.trajectories)} trajectories extracted")
    plt.tight_layout(); plt.show()

    for i, t in enumerate(res.trajectories):
        print(f"  traj {i}: length={t.length_px:.1f}px  speed={t.speed_px_per_frame:.1f}/frame  rad={t.radius_px:.1f}  conf={t.confidence:.3f}")
